In [ ]:
from langchain_ollama import OllamaEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS

In [ ]:
def get_embeddings():
    return OllamaEmbeddings(model="nomic-embed-text")

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
def load_vector_db():
    return FAISS.load_local(
        "faiss_books_index",
        get_embeddings(),
        allow_dangerous_deserialization=True
    )

In [ ]:
def get_qa_chain(vectorstore):
    retriever = vectorstore.as_retriever(
        search_type="mmr",  #  better than similarity
        search_kwargs={"k": 5, "fetch_k": 10}
    )

    llm = ChatGroq(
        temperature=0,
        model="openai/gpt-oss-120b"
    )

    return retriever, llm

In [ ]:
def rewrite_query(query, llm):
    prompt = f"""
    Rewrite the user query to be more specific and optimized for semantic search.

    User Query: {query}

    Improved Query:
    """
    return llm.invoke(prompt)

In [ ]:
def ask_question(query, retriever, llm):
    #  Step 1: Rewrite query
    # improved_query = rewrite_query(query, llm)

    # Step 2: Retrieve documents
    docs = retriever.invoke(query)

    #  Step 3: Build context
    context = "\n\n".join([doc.page_content for doc in docs])

    # Step 4: Final prompt
    prompt = f"""
    Answer the question using ONLY the context below.
    If answer not found, say "I don't know".

    Context:
    {context}

    Question:
    {query}

    Answer:
    """

    answer = llm.invoke(prompt)

    # Step 5: Source citations
    sources = []
    for doc in docs:
        sources.append({
            "title": doc.metadata.get("title"),
            "url": doc.metadata.get("url")
        })

    return answer, sources

In [ ]:
vectorstore = load_vector_db()

retriever, llm = get_qa_chain(vectorstore)

while True:
    query = input("\n Ask a question: ")

    answer, sources = ask_question(query, retriever, llm)

    print("\n Answer:")
    print(answer)

    print("\n Sources:")
    for s in sources:
        print(f"{s['title']} → {s['url']}")